In [0]:
# 1. Configuración

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "si7006_t2"
TBL_BRONZE      = f"{CATALOG}.{SCHEMA}.orders_bronze"
TBL_SILVER      = f"{CATALOG}.{SCHEMA}.orders_silver"
VOL_CHECKPOINTS = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"
CKPT_SILVER     = f"{VOL_CHECKPOINTS}/silver"

WATERMARK_DELAY = "24 hours"

print(f"Bronze:     {TBL_BRONZE}")
print(f"Silver:     {TBL_SILVER}")
print(f"Checkpoint: {CKPT_SILVER}")

Bronze:     workspace.si7006_t2.orders_bronze
Silver:     workspace.si7006_t2.orders_silver
Checkpoint: /Volumes/workspace/si7006_t2/checkpoints/silver


In [0]:
# 2. Lectura streaming desde Bronze + transformaciones

df_silver = (
    spark.readStream
        .format("delta")
        .table(TBL_BRONZE)

        # Enriquecimiento
        .withColumn("event_time",   F.to_timestamp(F.col("InvoiceDate")))
        .withColumn("total_amount", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
        .withColumn("is_guest",     F.col("CustomerID").isNull())
        .withColumn("customer_id_imputed",
                    F.coalesce(F.col("CustomerID").cast("string"), F.lit("GUEST")))

        # Filtros de calidad
        .filter(~F.col("InvoiceNo").startswith("C"))
        .filter(F.col("Quantity") > 0)
        .filter(F.col("UnitPrice") > 0)
        .filter(F.col("StockCode").isNotNull() & (F.trim(F.col("StockCode")) != ""))
        .filter(F.col("event_time").isNotNull())

        # Watermark + deduplicación stateful por (InvoiceNo, StockCode)
        .withWatermark("event_time", WATERMARK_DELAY)
        .dropDuplicatesWithinWatermark(["InvoiceNo", "StockCode"])

        # Selección final con renombrado a snake_case
        .select(
            F.col("InvoiceNo").alias("invoice_no"),
            F.col("StockCode").alias("stock_code"),
            F.col("Description").alias("description"),
            F.col("Country").alias("country"),
            F.col("customer_id_imputed").alias("customer_id"),
            F.col("is_guest"),
            F.col("Quantity").alias("quantity"),
            F.col("UnitPrice").alias("unit_price"),
            F.col("total_amount"),
            F.col("event_time"),
            F.col("ingested_at").alias("bronze_ingested_at"),
            F.col("source_file").alias("bronze_source_file"),
        )
)

df_silver.printSchema()

root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: string (nullable = false)
 |-- is_guest: boolean (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)



In [0]:
# 3. Loop de escritura con Trigger.AvailableNow

import time

ITERACIONES    = 5
PAUSA_SEGUNDOS = 12

for i in range(1, ITERACIONES + 1):
    query = (
        df_silver.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", CKPT_SILVER)
            .option("mergeSchema", "false")
            .trigger(availableNow=True)
            .toTable(TBL_SILVER)
    )
    query.awaitTermination()

    total = spark.sql(f"SELECT COUNT(*) AS n FROM {TBL_SILVER}").collect()[0]["n"]
    print(f"[{i:02d}/{ITERACIONES}] {time.strftime('%H:%M:%S')} | total: {total:,}")

    if i < ITERACIONES:
        time.sleep(PAUSA_SEGUNDOS)

[01/5] 16:51:19 | total: 2,927
[02/5] 16:51:33 | total: 2,927
[03/5] 16:51:49 | total: 2,927
[04/5] 16:52:04 | total: 2,927
[05/5] 16:52:19 | total: 2,927


In [0]:
# 4. Validación: schema, métricas y duplicados

spark.table(TBL_SILVER).printSchema()

spark.sql(f"""
    SELECT COUNT(*)                                       AS total,
           SUM(CASE WHEN is_guest THEN 1 ELSE 0 END)      AS guests,
           ROUND(SUM(total_amount), 2)                    AS sum_amount,
           ROUND(AVG(total_amount), 2)                    AS avg_amount,
           COUNT(DISTINCT country)                        AS paises,
           COUNT(DISTINCT stock_code)                     AS productos,
           MIN(event_time)                                AS primer_evento,
           MAX(event_time)                                AS ultimo_evento
    FROM {TBL_SILVER}
""").show(truncate=False)

# Verificación de duplicados de la clave de negocio
spark.sql(f"""
    SELECT invoice_no, stock_code, COUNT(*) AS c
    FROM {TBL_SILVER}
    GROUP BY invoice_no, stock_code
    HAVING COUNT(*) > 1
""").show()

root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- is_guest: boolean (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)

+-----+------+----------+----------+------+---------+-----------------------+-----------------------+
|total|guests|sum_amount|avg_amount|paises|productos|primer_evento          |ultimo_evento          |
+-----+------+----------+----------+------+---------+-----------------------+-----------------------+
|2927 |771   |56489.82  |19.3      |32    |1464     |2026-04-28 15:37:53.773|2026-04-28 16:31:16.979|
+-----+------+----------+----------+------+---------+--------

In [0]:
# 5. Historial Delta (evidencia de commits y time travel)

spark.sql(f"DESCRIBE HISTORY {TBL_SILVER}").select(
    "version", "timestamp", "operation",
    "operationMetrics.numOutputRows"
).show(truncate=False)

+-------+-------------------+----------------+-------------+
|version|timestamp          |operation       |numOutputRows|
+-------+-------------------+----------------+-------------+
|5      |2026-04-29 16:51:18|STREAMING UPDATE|0            |
|4      |2026-04-29 16:51:11|STREAMING UPDATE|0            |
|3      |2026-04-28 20:55:32|OPTIMIZE        |NULL         |
|2      |2026-04-28 20:55:09|STREAMING UPDATE|0            |
|1      |2026-04-28 20:54:55|STREAMING UPDATE|2927         |
|0      |2026-04-28 20:54:33|CREATE TABLE    |NULL         |
+-------+-------------------+----------------+-------------+

